Week 3's "local storage" flow: pulls a NEW batch of COCO images (not
the same 60 from before), saves them as real files on disk in
local-store/ first, then loads from those local files into raw as a
genuine incremental snapshot (INSERT, not replace). This is the piece
the earlier ingestion scripts skipped -- they streamed straight from
HF into RustFS, never actually touching local Docker host storage.

Standard library imports for byte buffers, JSON encoding, and filesystem paths.

In [1]:
import io
import json
import os

Imports for S3/RustFS access, DuckDB, and streaming the dataset from Hugging Face.

In [2]:
import boto3
import duckdb
import pandas as pd
from datasets import load_dataset

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: how many new images to pull, how many to skip (already ingested), and the local staging directory.

In [3]:
N_NEW_IMAGES = 20
ALREADY_HAVE = 60   # skip the first 60, since those are already in raw.coco_annotations
BUCKET = "lakehouse"
S3_PREFIX = "assets/coco/images"
LOCAL_DIR = "/data/local/coco_staging"   # the actual local-store/ bind mount

Creates a boto3 S3 client pointed at RustFS.

In [4]:
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url="http://rustfs:9000",
        aws_access_key_id="rustfsadmin",
        aws_secret_access_key="rustfsadmin",
    )

Connects DuckDB and attaches the DuckLake catalog.

In [5]:
def attach_lakehouse():
    con = duckdb.connect()
    con.execute(open("sql/00_attach.sql").read())
    return con

Step 1: downloads new images from HF and saves them as real files on local disk — nothing touches RustFS or DuckLake yet.

In [6]:
# step 1: download from HF and save as real files on local disk --
# nothing touches RustFS or DuckLake yet, this is purely local staging
def download_to_local_store():
    os.makedirs(LOCAL_DIR, exist_ok=True)
    print(f"Streaming {N_NEW_IMAGES} NEW images (skipping the first {ALREADY_HAVE} already ingested)...")

    ds = load_dataset("ariG23498/coco2017", split="validation", streaming=True)
    ds = ds.skip(ALREADY_HAVE)   # get past the images we already have

    saved = []
    for i, sample in enumerate(ds):
        if i >= N_NEW_IMAGES:
            break

        img = sample["image"].convert("RGB")
        objects = sample["objects"]
        global_index = ALREADY_HAVE + i   # keeps filenames unique against the earlier batch

        local_path = f"{LOCAL_DIR}/{global_index:04d}.jpg"
        img.save(local_path, format="JPEG", quality=90)

        saved.append({
            "local_path": local_path,
            "global_index": global_index,
            "width": img.width,
            "height": img.height,
            "bbox_json": json.dumps(objects.get("bbox", [])),
            "segmentation_json": json.dumps(objects.get("segmentation", [])),
            "categories_json": json.dumps(objects.get("categories", [])),
        })

    print(f"Saved {len(saved)} images to {LOCAL_DIR} (visible in your local-store/ folder)")
    return saved

Step 2: uploads the staged local files to RustFS and inserts their metadata into `raw.coco_annotations`, growing the table instead of replacing it.

In [7]:
# step 2: now actually move the staged local files into the lakehouse --
# upload each to RustFS, then INSERT (not replace) the metadata into raw
def load_from_local_store_into_lakehouse(s3, con, staged):
    rows = []
    for item in staged:
        key = f"{S3_PREFIX}/{item['global_index']:04d}.jpg"
        with open(item["local_path"], "rb") as f:
            s3.put_object(Bucket=BUCKET, Key=key, Body=f, ContentType="image/jpeg")

        rows.append({
            "image_uri": f"s3://{BUCKET}/{key}",
            "width": item["width"],
            "height": item["height"],
            "bbox_json": item["bbox_json"],
            "segmentation_json": item["segmentation_json"],
            "categories_json": item["categories_json"],
        })

    df = pd.DataFrame(rows)
    con.register("new_coco_df", df)

    # INSERT, not CREATE OR REPLACE -- this is what makes it genuinely
    # incremental, growing the table instead of rebuilding it
    con.execute("INSERT INTO raw.coco_annotations SELECT * FROM new_coco_df")

    total = con.sql("SELECT COUNT(*) FROM raw.coco_annotations").fetchone()[0]
    print(f"raw.coco_annotations now has {total} rows (added {len(rows)} via local storage)")

Connects to RustFS and attaches the DuckLake catalog.

In [8]:
s3 = make_s3_client()

con = attach_lakehouse()

Downloads the new batch of images to local disk staging (Step 1).

In [9]:
staged = download_to_local_store()

Streaming 20 NEW images (skipping the first 60 already ingested)...


Saved 20 images to /data/local/coco_staging (visible in your local-store/ folder)


Uploads the staged files to RustFS and inserts their metadata into `raw.coco_annotations` (Step 2).

In [10]:
load_from_local_store_into_lakehouse(s3, con, staged)

raw.coco_annotations now has 80 rows (added 20 via local storage)


Shows the most recent DuckLake snapshots as proof a new version was created.

In [11]:
print("\nMost recent snapshots:")

con.sql("FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 5").show()


Most recent snapshots:
┌─────────────┬───────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │                              changes                              │ author  │ commit_message │ commit_extra_info │
│    int64    │   timestamp with time zone    │     int64      │                      map(varchar, varchar[])                      │ varchar │    varchar     │      varchar      │
├─────────────┼───────────────────────────────┼────────────────┼───────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           5 │ 2026-08-09 02:29:38.660409+00 │              4 │ {tables_inserted_into=[4]}                                        │ NULL    │ NULL           │ NULL              │
│           4 │ 2026-08-09 02:29:26.802554+00 │              4 │ {tables_cre